# Fine-tune BARTpho đặt tên sự kiện (so với ViT5)

Notebook chạy trên **Kaggle** (GPU **T4 x1**, bật **Internet**), cùng Kaggle Dataset `pbl-naming-v1` với notebook ViT5
(`train.jsonl` 2.751 + `dev.jsonl` 320 của `data/naming/v1/`).

**Mô hình:** `BART_MODEL` ở ô cấu hình — `vinai/bartpho-syllable` (BART-large, 12+12 lớp, d_model 1024, ~393M tham số) hoặc
`vinai/bartpho-syllable-base` (6+6 lớp, d_model 768, ~130M). Bản *syllable* đọc thẳng âm tiết như ViT5, không cần tách từ;
bản *word* bắt buộc VnCoreNLP (Java) và không khớp pyvi → không dùng. Khác ViT5: BARTpho chỉ tiền huấn luyện khử nhiễu
(chưa học tóm tắt), nên "zero-shot" chỉ là mốc tham khảo.

**Tokenizer:** dùng `tokenizer.json` (fast) của VinAI qua `PreTrainedTokenizerFast` — không cần sentencepiece. Đo 04/11 trên
1.000 câu của bộ dữ liệu: bản slow (`BartphoTokenizer`) ra 203 token `<unk>` (âm tiết ngoài từ điển 40k như "hòa", "dọa"),
bản fast tách thành mảnh nhỏ, 0 `<unk>`; 881/1.000 câu trùng mã.

**Luồng giống hệt notebook ViT5** (cùng làm sạch đầu vào, bộ chấm ROUGE theo âm tiết + bootstrap, mốc medoid, dừng sớm theo
ROUGE-L dev, 3 seed) để so được. So trực tiếp với ViT5 làm ở máy bằng dự đoán dev của hai mô hình (bootstrap ghép cặp).
**Dev là nhãn LLM — chỉ để chọn mô hình, không báo cáo như kết quả thật.**

## 0. Cấu hình

- `lr`: 3e-5 cho bản lớn, 5e-5 cho bản base (mức thường dùng khi fine-tune BART; ViT5 dùng 1e-4 vì họ T5 cần lr cao hơn).
- `fp16=False`: giữ fp32 như ViT5 để so công bằng và tránh loss NaN.
- `max_source_len` 512 token: đầu vào sau làm sạch p99 ≈ 380–440 token, dài nhất ~510 (đo 03/11) → gần như không
  mẫu nào bị cắt; ô dữ liệu in tỉ lệ bị cắt.
- `max_target_len` 48: tên tối đa 20 âm tiết ≈ 30 token.
- `patience` 2: dừng khi ROUGE-L dev không tăng 2 epoch liền; giữ checkpoint tốt nhất theo dev.

In [ ]:
import os

BART_MODEL = "vinai/bartpho-syllable"     # vinai/bartpho-syllable | vinai/bartpho-syllable-base
TAG = "bartpho-naming"                   # tiền tố tên mô hình / file kết quả
CFG = dict(
    model_name=BART_MODEL,
    seeds=[42, 43, 44],
    lr=3e-5, epochs=8, batch_size=8, grad_accum=2, warmup_ratio=0.05, weight_decay=0.01, max_grad_norm=1.0,
    max_source_len=512, max_target_len=48,
    fp16=False,
    patience=2,
    num_beams=4, max_new_tokens=48, no_repeat_ngram_size=3, eval_batch_size=16,
    max_titles=12, max_sapos=3,          # sau làm sạch: tối đa 12 tiêu đề (medoid đứng đầu) + 3 sapo
    max_syllables=20,
    bootstrap=1000,
)
OUT = __import__("pathlib").Path("/kaggle/working" if os.path.exists("/kaggle/working") else "kaggle_out")
OUT.mkdir(parents=True, exist_ok=True)

## 1. Môi trường

In [ ]:
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import math, random, re, time, unicodedata
from collections import Counter

import numpy as np
import torch
import transformers
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          get_linear_schedule_with_warmup)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "· transformers", transformers.__version__, "· thiết bị", device,
      torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def load_tokenizer(name):
    # tokenizer.json (fast) của VinAI: không cần sentencepiece, không sinh <unk> cho âm tiết ngoài từ điển 40k.
    from huggingface_hub import hf_hub_download
    from transformers import PreTrainedTokenizerFast
    return PreTrainedTokenizerFast(tokenizer_file=hf_hub_download(name, "tokenizer.json"), bos_token="<s>",
                                   eos_token="</s>", unk_token="<unk>", pad_token="<pad>", mask_token="<mask>")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 2. Nạp dữ liệu và làm sạch đầu vào

`input` trong jsonl là khối văn bản mà các agent gán nhãn đã đọc: dòng đầu `## <event_id> · n bài · n báo · ngày`,
mỗi tiêu đề có `[báo ngày giờ]` và `(medoid)`, rồi các dòng sapo. Mô hình không cần mã sự kiện, tên báo, giờ đăng
(dễ học chép ngày vào tên) → **làm sạch** thành:

`Tiêu đề: <medoid> | <tiêu đề 2> | … . Tóm tắt: <sapo 1> | <sapo 2> | …`

Tiêu đề trùng nhau (khác báo đăng y hệt) chỉ giữ một. **Lúc chạy thật phải dựng đầu vào đúng y như hàm `clean_input`**
(medoid đứng đầu, tối đa `max_titles` tiêu đề, `max_sapos` sapo).

In [ ]:
import json
from pathlib import Path

NEED = {"train.jsonl", "dev.jsonl"}
if Path("/kaggle/input").exists():
    _dirs = [Path(d) for d, _, fs in os.walk("/kaggle/input", followlinks=True) if NEED <= set(fs)]
    if not _dirs:
        raise FileNotFoundError("Không thấy train.jsonl + dev.jsonl trong /kaggle/input — đã Add Input dataset chưa?")
    DATA = _dirs[0]
else:
    DATA = Path("data/naming/v1")
print("dữ liệu:", DATA)

load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
train_raw, dev_raw = load(DATA / "train.jsonl"), load(DATA / "dev.jsonl")

_TITLE = re.compile(r"^\d+\. \[[^\]]*\] (.*?)( \(medoid\))?$")
_SAPO = re.compile(r"^- \(\d+\) (.*)$")
nfc = lambda s: unicodedata.normalize("NFC", s).strip()


def parse_block(block):
    # → (danh sách tiêu đề theo thời gian, chỉ số medoid hoặc None, danh sách sapo)
    titles, medoid, sapos = [], None, []
    for line in block.splitlines():
        m = _TITLE.match(line)
        if m:
            if m.group(2):
                medoid = len(titles)
            titles.append(nfc(m.group(1)))
            continue
        m = _SAPO.match(line)
        if m:
            sapos.append(nfc(m.group(1)))
    return titles, medoid, sapos


def clean_input(block, max_titles=CFG["max_titles"], max_sapos=CFG["max_sapos"]):
    titles, medoid, sapos = parse_block(block)
    order = ([medoid] if medoid is not None else []) + [i for i in range(len(titles)) if i != medoid]
    seen, kept = set(), []
    for i in order:
        key = titles[i].lower()
        if key not in seen:
            seen.add(key)
            kept.append(titles[i])
    text = "Tiêu đề: " + " | ".join(kept[:max_titles]) + " ."
    if sapos:
        text += " Tóm tắt: " + " | ".join(sapos[:max_sapos])
    return text


def medoid_title(block):
    titles, medoid, _ = parse_block(block)
    return titles[medoid if medoid is not None else 0]


def prep(rows):
    return [dict(event_id=r["event_id"], src=clean_input(r["input"]), tgt=nfc(r["target"]),
                 medoid=medoid_title(r["input"]), raw=r["input"]) for r in rows]


train, dev = prep(train_raw), prep(dev_raw)
assert not ({r["event_id"] for r in train} & {r["event_id"] for r in dev}), "train và dev trùng sự kiện"
print(f"train {len(train)} · dev {len(dev)}")
print("VÍ DỤ đầu vào:", train[0]["src"][:400], "\nVÍ DỤ đích:", train[0]["tgt"])

tok = load_tokenizer(CFG["model_name"])
_ids = tok("a").input_ids
assert _ids[0] == tok.bos_token_id and _ids[-1] == tok.eos_token_id, "tokenizer không thêm <s> … </s>"
for name, rows in (("train", train), ("dev", dev)):
    ls = np.array([len(tok(r["src"]).input_ids) for r in rows])
    lt = np.array([len(tok(r["tgt"]).input_ids) for r in rows])
    print(f"{name}: token đầu vào trung vị {np.median(ls):.0f}, p99 {np.percentile(ls, 99):.0f}, max {ls.max()}, "
          f"bị cắt ở {CFG['max_source_len']}: {(ls > CFG['max_source_len']).mean():.1%} · "
          f"token đích p99 {np.percentile(lt, 99):.0f}, max {lt.max()} (cắt ở {CFG['max_target_len']})")

## 3. Bộ chấm

- **ROUGE-1/2/L F1 theo âm tiết** (NFC, chữ thường, bỏ dấu câu, tách khoảng trắng). Không dùng `rouge_score` mặc định:
  bộ tách từ của nó bỏ mọi ký tự ngoài a–z → mất dấu tiếng Việt.
- **Độ dài**: số âm tiết trung bình, % tên ≤ 20 âm tiết.
- **Trung thực (xấp xỉ)**: % tên mà mọi con số và mọi chữ viết hoa giữa câu (tên riêng) đều có trong đầu vào. Đây là
  phép kiểm thô (không NER, không alias) — chỉ để so tương đối giữa các mô hình.
- **CI 95%**: bootstrap theo sự kiện; so hai hệ thống bằng bootstrap ghép cặp trên hiệu ROUGE-L.

In [ ]:
_PUNCT = re.compile(r"[^\w\s]")
_NUM = re.compile(r"\d+(?:[.,/:-]\d+)*")


def syl(s):
    return _PUNCT.sub(" ", unicodedata.normalize("NFC", s).lower()).split()


def _ngrams(t, n):
    return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1))


def _f1(overlap, n_pred, n_ref):
    if not n_pred or not n_ref or not overlap:
        return 0.0
    p, r = overlap / n_pred, overlap / n_ref
    return 2 * p * r / (p + r)


def _lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]


def rouge(pred, ref):
    p, r = syl(pred), syl(ref)
    out = {}
    for n in (1, 2):
        a, b = _ngrams(p, n), _ngrams(r, n)
        out[f"rouge{n}"] = _f1(sum((a & b).values()), sum(a.values()), sum(b.values()))
    out["rougeL"] = _f1(_lcs(p, r), len(p), len(r))
    return out


def faithful(pred, src):
    low = unicodedata.normalize("NFC", src).lower()
    nums_ok = all(n in src for n in _NUM.findall(pred))
    words = pred.split()[1:]                                     # bỏ chữ đầu câu (luôn viết hoa)
    caps = [w.strip(",.;:!?()\"'“”") for w in words if w[:1].isupper()]
    return nums_ok and all(w.lower() in low for w in caps if w)


def per_item(preds, rows):
    out = []
    for p, r in zip(preds, rows):
        d = rouge(p, r["tgt"])
        n = len(p.split())
        d.update(len=n, le_max=float(n <= CFG["max_syllables"]), faithful=float(faithful(p, r["src"])))
        out.append(d)
    return out


def boot_ci(x, B=CFG["bootstrap"], seed=0):
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, len(x), (B, len(x)))].mean(1)
    return float(x.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def summarize(items):
    keys = ("rouge1", "rouge2", "rougeL", "len", "le_max", "faithful")
    return {k: boot_ci([it[k] for it in items]) for k in keys}


def show(name, s):
    f = lambda k: f"{s[k][0]:.3f} [{s[k][1]:.3f}; {s[k][2]:.3f}]"
    print(f"{name:28s} R1 {f('rouge1')} · R2 {s['rouge2'][0]:.3f} · RL {f('rougeL')} · "
          f"dài {s['len'][0]:.1f} âm tiết · ≤20: {s['le_max'][0]:.0%} · trung thực {s['faithful'][0]:.0%}")


def paired(a_items, b_items, key="rougeL"):
    d = np.array([a[key] - b[key] for a, b in zip(a_items, b_items)])
    return boot_ci(d)


assert abs(rouge("Tuyển nữ Việt Nam thua Đài Loan 1-2", "tuyển nữ việt nam thua đài loan 1-2")["rougeL"] - 1) < 1e-9

## 4. Mốc so sánh trên dev

- **Tiêu đề medoid** nguyên văn (baseline bắt buộc của đề tài, cổng 7: mô hình phải thắng).
- **Medoid rút gọn bằng luật**: bỏ tiền tố ngắn trước dấu hai chấm ("Video:", "Hà Nội:"), bỏ ngoặc kép và dấu hỏi,
  cắt còn ≤ 20 âm tiết.
- **BARTpho chưa fine-tune** (cùng cách sinh) — chỉ tham khảo: mô hình khử nhiễu chưa học tóm tắt nên thường chép đầu vào.

In [ ]:
def shorten(title, k=CFG["max_syllables"]):
    t = title
    if ":" in t:
        left, right = t.split(":", 1)
        t = right if len(left.split()) <= 3 else left + "," + right
    t = re.sub(r"[\"'“”‘’?]", "", t).strip(" .,-")
    w = t.split()
    return " ".join(w[:k]).rstrip(",;-") if len(w) > k else t


@torch.no_grad()
def generate(model, rows, bs=CFG["eval_batch_size"]):
    model.eval()
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]["src"]))   # gom độ dài giống nhau → ít padding
    preds = [None] * len(rows)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok([rows[i]["src"] for i in idx], max_length=CFG["max_source_len"], truncation=True,
                  padding=True, return_tensors="pt").to(device)
        out = model.generate(**enc, num_beams=CFG["num_beams"], max_new_tokens=CFG["max_new_tokens"],
                             no_repeat_ngram_size=CFG["no_repeat_ngram_size"], early_stopping=True)
        for i, text in zip(idx, tok.batch_decode(out, skip_special_tokens=True)):
            preds[i] = nfc(text)
    return preds


baselines = {}
base_items = {}
for name, preds in [("medoid", [r["medoid"] for r in dev]), ("medoid_rut_gon", [shorten(r["medoid"]) for r in dev])]:
    base_items[name] = per_item(preds, dev)
    baselines[name] = summarize(base_items[name])
    show(name, baselines[name])

t0 = time.time()
_zs = AutoModelForSeq2SeqLM.from_pretrained(CFG["model_name"]).to(device)
zs_preds = generate(_zs, dev)
base_items["zero_shot"] = per_item(zs_preds, dev)
baselines["zero_shot"] = summarize(base_items["zero_shot"])
show("zero_shot", baselines["zero_shot"])
print(f"(sinh {len(dev)} tên mất {time.time() - t0:.0f}s) ví dụ:", zs_preds[:3])
del _zs
torch.cuda.empty_cache()

## 5. Huấn luyện

Vòng lặp PyTorch thuần (không dùng `Seq2SeqTrainer` để chạy được cả transformers 4.x lẫn 5.x): AdamW + warmup tuyến
tính, tích luỹ gradient, cắt gradient 1,0. Hết mỗi epoch sinh tên cho dev, tính ROUGE-L; lưu checkpoint khi ROUGE-L
tăng, dừng sau `patience` epoch không tăng. Loss NaN → dừng ngay (bật `fp16` thì tắt đi).

In [ ]:
import csv
import shutil

def encode(rows):
    enc = tok([r["src"] for r in rows], max_length=CFG["max_source_len"], truncation=True)
    lab = tok(text_target=[r["tgt"] for r in rows], max_length=CFG["max_target_len"], truncation=True)
    return [dict(input_ids=enc["input_ids"][i], attention_mask=enc["attention_mask"][i], labels=lab["input_ids"][i])
            for i in range(len(rows))]


train_feats = encode(train)


def train_one(seed):
    set_seed(seed)
    tag = f"{TAG}-s{seed}"
    model = AutoModelForSeq2SeqLM.from_pretrained(CFG["model_name"]).to(device)
    collate = DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100)
    loader = torch.utils.data.DataLoader(train_feats, batch_size=CFG["batch_size"], shuffle=True, collate_fn=collate,
                                         generator=torch.Generator().manual_seed(seed))
    no_decay = ("bias", "layer_norm.weight", "layernorm_embedding.weight")
    groups = [{"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)],
               "weight_decay": CFG["weight_decay"]},
              {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=CFG["lr"])
    total = math.ceil(len(loader) / CFG["grad_accum"]) * CFG["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(total * CFG["warmup_ratio"]), total)
    use_amp = CFG["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best, bad, log, best_preds = -1.0, 0, [], None
    for ep in range(1, CFG["epochs"] + 1):
        model.train()
        t0, run_loss, n = time.time(), 0.0, 0
        for i, batch in enumerate(loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise FloatingPointError(f"loss = {loss.item()} ở epoch {ep}, bước {i} — đặt CFG['fp16']=False")
            scaler.scale(loss / CFG["grad_accum"]).backward()
            run_loss, n = run_loss + loss.item(), n + 1
            if (i + 1) % CFG["grad_accum"] == 0 or i + 1 == len(loader):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"])
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
        preds = generate(model, dev)
        items = per_item(preds, dev)
        rl = float(np.mean([it["rougeL"] for it in items]))
        log.append(dict(epoch=ep, train_loss=run_loss / n, dev_rougeL=rl,
                        dev_len=float(np.mean([it["len"] for it in items])), minutes=(time.time() - t0) / 60))
        print(f"[{tag}] epoch {ep}: loss {run_loss / n:.4f} · dev ROUGE-L {rl:.4f} · "
              f"dài {log[-1]['dev_len']:.1f} · {log[-1]['minutes']:.1f} phút")
        if rl > best:
            best, bad, best_preds, best_items = rl, 0, preds, items
            path = OUT / "models" / tag
            shutil.rmtree(path, ignore_errors=True)
            model.save_pretrained(path)
            tok.save_pretrained(path)
        else:
            bad += 1
            if bad >= CFG["patience"]:
                print(f"[{tag}] dừng sớm sau epoch {ep}")
                break
    with open(OUT / f"du_doan_dev_{tag}.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow(["event_id", "ten_nhan", "ten_mo_hinh", "medoid", "rougeL"])
        for r, p, it in zip(dev, best_preds, best_items):
            w.writerow([r["event_id"], r["tgt"], p, r["medoid"], f"{it['rougeL']:.3f}"])
    del model, opt
    torch.cuda.empty_cache()
    return dict(tag=tag, seed=seed, log=log, best_epoch=max(log, key=lambda x: x["dev_rougeL"])["epoch"],
                dev=summarize(best_items), items=best_items, preds=best_preds)


runs = []
for seed in CFG["seeds"]:
    runs.append(train_one(seed))

## 6. Kết quả

Cách đọc:
- So với **medoid** (baseline bắt buộc): hiệu ROUGE-L > 0 và **CI không chứa 0** → mô hình gần nhãn hơn medoid. Đây
  vẫn là nhãn LLM — kết luận thật cần tập test tên người viết mù + so cặp mù (cổng 7).
- So với **BARTpho chưa fine-tune**: chỉ tham khảo.
- **≤ 20 âm tiết** phải gần 100%; **trung thực** thấp hơn medoid nhiều → mô hình bịa số / tên riêng, xem file dự đoán.
- 3 seed: trung bình ± độ lệch chuẩn; chênh giữa seed lớn → kết quả chưa ổn định.

In [ ]:
for name, s in baselines.items():
    show(name, s)
for r in runs:
    show(f"{r['tag']} (ep {r['best_epoch']})", r["dev"])
    for b in ("medoid", "medoid_rut_gon", "zero_shot"):
        m, lo, hi = paired(r["items"], base_items[b])
        print(f"{'':30s} ΔRL so {b:15s} {m:+.3f} [{lo:+.3f}; {hi:+.3f}]")

summary = {}
for key in ("rouge1", "rouge2", "rougeL", "len", "le_max", "faithful"):
    v = np.array([r["dev"][key][0] for r in runs])
    summary[key] = dict(mean=float(v.mean()), std=float(v.std(ddof=1)) if len(v) > 1 else 0.0)
    print(f"{key:9s} qua {len(runs)} seed: {v.mean():.3f} ± {summary[key]['std']:.3f}"
          f" (medoid {baselines['medoid'][key][0]:.3f}, zero-shot {baselines['zero_shot'][key][0]:.3f})")

best = max(runs, key=lambda r: r["dev"]["rougeL"][0])
results = dict(
    config=CFG, data=str(DATA), n_train=len(train), n_dev=len(dev),
    versions=dict(torch=torch.__version__, transformers=transformers.__version__),
    note="dev = nhãn LLM (3 vòng), chỉ để chọn cấu hình; chưa phải kết quả báo cáo",
    baselines=baselines,
    runs=[{k: v for k, v in r.items() if k not in ("items", "preds")} |
          {"vs": {b: paired(r["items"], base_items[b]) for b in base_items}} for r in runs],
    summary=summary, best_tag=best["tag"])
(OUT / f"results_{TAG}.json").write_text(json.dumps(results, ensure_ascii=False, indent=1), encoding="utf-8")

print("\nVí dụ (mô hình tốt nhất theo dev):")
for r, p in list(zip(dev, best["preds"]))[:15]:
    print(f"- nhãn : {r['tgt']}\n  model: {p}\n  medoid: {r['medoid']}")

## 7. Lấy kết quả về

- **Save Version → Save & Run All** để chạy nền (3 seed × tối đa 8 epoch, ước ~1–2 giờ trên T4 với fp32); khi xong,
  tab *Output* có `models/<tag>/`, `results_<TAG>.json`, `du_doan_dev_<tag>.csv` và `best_model_<tag>.zip`.
- Tải về, giải nén mô hình vào `data/models/<tag>_<ngày>/`; JSON + CSV vào
  `data/temp/kaggle_runs/<ngày>_<TAG>/`.
- Ô dưới xoá các mô hình không tốt nhất để Output nhẹ (bản lớn ~1,6 GB, bản base ~0,5 GB) và nén mô hình tốt nhất (theo dev).

In [ ]:
for r in runs:
    if r["tag"] != best["tag"]:
        shutil.rmtree(OUT / "models" / r["tag"], ignore_errors=True)
zip_path = shutil.make_archive(str(OUT / f"best_model_{best['tag']}"), "zip", root_dir=OUT / "models",
                               base_dir=best["tag"])
print(f"mô hình chọn theo dev ROUGE-L: {best['tag']} ({best['dev']['rougeL'][0]:.3f})")
for f in sorted(OUT.iterdir()):
    size = sum(x.stat().st_size for x in f.rglob("*")) if f.is_dir() else f.stat().st_size
    print(f"  {f.name:45s} {size / 1e6:8.1f} MB")